In [1]:
import os
import pickle
from pathlib import Path

import faiss
import numpy as np
from pypdf import PdfReader
from openai import OpenAI

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
PDF_PATH = r"2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"
INDEX_PATH = "indigo_faiss.index"
META_PATH = "indigo_chunks.pkl"

EMBED_MODEL = "text-embedding-3-small"
GEN_MODEL = "gpt-5.4"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def load_pdf_text(pdf_path: str) -> list[dict]:
    reader = PdfReader(pdf_path)
    docs = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            docs.append({
                "page": i + 1,
                "text": text.strip()
            })
    return docs

In [4]:
def split_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    text = " ".join(text.split())
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break
        start = end - overlap

    return chunks

In [5]:
def build_chunks(docs: list[dict]) -> list[dict]:
    all_chunks = []

    for doc in docs:
        page = doc["page"]
        text = doc["text"]

        chunks = split_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "page": page,
                "chunk_id": idx,
                "text": chunk
            })

    return all_chunks

In [6]:
def embed_texts(texts: list[str], batch_size: int = 64) -> np.ndarray:
    vectors = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(
            model=EMBED_MODEL,
            input=batch
        )
        batch_vecs = [item.embedding for item in resp.data]
        vectors.extend(batch_vecs)

    return np.array(vectors, dtype="float32")

In [7]:
def build_and_save_index(pdf_path: str):
    docs = load_pdf_text(pdf_path)
    chunks = build_chunks(docs)

    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)
    with open(META_PATH, "wb") as f:
        pickle.dump(chunks, f)

    print(f"인덱스 저장 완료: {INDEX_PATH}")
    print(f"청크 저장 완료: {META_PATH}")
    print(f"총 청크 수: {len(chunks)}")

In [8]:
def load_index_and_meta():
    if not Path(INDEX_PATH).exists() or not Path(META_PATH).exists():
        raise FileNotFoundError("먼저 build_and_save_index()를 실행하세요.")

    index = faiss.read_index(INDEX_PATH)
    with open(META_PATH, "rb") as f:
        chunks = pickle.load(f)

    return index, chunks

In [9]:
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    index, chunks = load_index_and_meta()

    q_emb = client.embeddings.create(
        model=EMBED_MODEL,
        input=[query]
    )
    q_vec = np.array([q_emb.data[0].embedding], dtype="float32")

    distances, indices = index.search(q_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        item = chunks[idx].copy()
        item["score"] = float(distances[0][rank])
        results.append(item)

    return results

In [10]:
def answer_question(query: str, top_k: int = TOP_K) -> str:
    retrieved = retrieve(query, top_k=top_k)

    context_blocks = []
    for i, item in enumerate(retrieved, 1):
        context_blocks.append(
            f"[근거 {i}] (page {item['page']})\n{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
너는 업로드된 PDF 문서만 근거로 답하는 한국어 경제 보고서 QA 도우미다.

규칙:
- 반드시 아래 검색 문맥에 근거해서만 답하라.
- 문맥에 없으면 "문서에서 확인되지 않습니다."라고 말하라.
- 답변 마지막에 참고한 page 번호를 적어라.
- 핵심만 한국어로 정리하라.

[사용자 질문]
{query}

[검색 문맥]
{context}
"""

    resp = client.responses.create(
        model=GEN_MODEL,
        input=prompt
    )

    return resp.output_text

In [11]:
build_and_save_index(PDF_PATH)

인덱스 저장 완료: indigo_faiss.index
청크 저장 완료: indigo_chunks.pkl
총 청크 수: 164


In [12]:
results = retrieve("2026년 한국 GDP 성장률 전망은?", top_k=4)

for r in results:
    print(f"page={r['page']}, score={r['score']:.4f}")
    print(r["text"][:300])
    print("-" * 80)

page=10, score=0.7976
< 요약 4/8 > 6 경제전망  금년 성장률은 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 개선세 확대, 예상보다 양호한 세계경제 흐름 등에 힘입어 지난 11월 전망수준보다 높은 2.0%를 나타낼 것으로 예상된다. ▪ 1/4분기중에는 소비가 회복세를 지속하고 수출도 반도체를 중심으로 강한 증가 세가 나타나는 데다, 전분기 역성장-0.3%의 기저효과주로 투자부문도 작용하면서 성장 률이 당초 예상0.3%을 상당폭 상회하여 1%에 근접0.9%할 전망이다. ▪ 2/4분기 이후에도 소득여건 개선 등으로 소비 회복세가 완만
--------------------------------------------------------------------------------
page=44, score=0.8113
30 <경제성장 전망1)> (전년동기대비, %) 2024 2025 2026e) 2027e) 연간 상반 하반 연간 상반 하반 연간 연간 GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 1.8 <0.3> <1.8> <1.0> <2.2> <1.5> <1.8> <1.9> • 민간소비 1.1 0.7 1.9 1.3 2.3 1.3 1.8 1.8 <0.7> <1.9> <1.3> <2.2> <1.3> <1.7> <1.7> • 건설투자 -3.3 -12.2 -7.5 -9.9 -0.8 2.6 1.0 1.5 <-12.4> <-5.3> <-
--------------------------------------------------------------------------------
page=80, score=0.8162
66 한국은행 전망에서는 점진적∙보수적인 전망 경향이 나타남 [그림3] 분기별 GDP 성장률1) 및 전망 [그림4] 분기성장률(전년동기대비) 증감과 전망오차2) 주: 1) 속보치 기준 2) 성장률이 전기보다 확대되는 경우 과소추정, 축소되는 경우 과대추정하는 경향을 나타냄 자료: 저자 계산 7. 최근 전

In [13]:
answer = answer_question("2026년 한국 GDP 성장률 전망은 몇 퍼센트야?")
print(answer)

2026년 한국 GDP 성장률 전망은 **2.0%**입니다.  
참고: page 10, page 44


In [14]:
while True:
    q = input("\n질문 입력 (종료: exit) > ").strip()
    if q.lower() == "exit":
        break

    try:
        ans = answer_question(q, top_k=4)
        print("\n[답변]")
        print(ans)

        print("\n[검색된 근거]")
        for r in retrieve(q, top_k=4):
            print(f"- page {r['page']} | score={r['score']:.4f}")
            print(r["text"][:200], "...")
            print()

    except Exception as e:
        print(f"오류 발생: {e}")